# Predicting Infertility (and 5 other conditions) from the Daily Symptom Log

**Input:** `AI_MSHM_Active_Input_Log.csv` - 28 daily rows per patient (1,082 patients) of symptom scores.
**Labels:** the log contains no disease outcomes, so targets come from `symptom_intensity_predictions_full.csv` (`<Disease>_ActualFlag`, `<Disease>_ActualScore`), joined on `Patient_File_No`.

For each of Infertility, Dysmenorrhea, PMDD, T2D, CVD and Endometrial:
- **XGBClassifier** on the 0/1 flag, with severity categories from predicted probability
- **XGBRegressor** on the 0-1 risk score, with severity categories from the score

Per-patient predictions are **out-of-fold** (cross-validated). Final models are refit on all patients and saved as `.pkl` and `.joblib`.

> `PCOS_Status_Label` is **excluded from the features**. In the label file, `Infertility_ActualFlag` is identical to the PCOS label for every patient, so including it would make the Infertility model trivially perfect.

In [1]:
import os, warnings, pickle, joblib
import numpy as np, pandas as pd
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score,
                             confusion_matrix, r2_score, mean_absolute_error, mean_squared_error)
from openpyxl.styles import Font
warnings.filterwarnings("ignore")

LOG   = "/mnt/user-data/uploads/AI_MSHM_Active_Input_Log.csv"
LABEL = "/mnt/user-data/uploads/symptom_intensity_predictions_full.csv"
OUT   = "/mnt/user-data/outputs/symptom_log_model"
os.makedirs(f"{OUT}/models", exist_ok=True)
SEED = 42
DISEASES = ["Infertility", "Dysmenorrhea", "PMDD", "T2D", "CVD", "Endometrial"]

## 1. Load the daily log

In [2]:
log = pd.read_csv(LOG)
print(log.shape, "| patients:", log.Patient_File_No.nunique(), "| missing:", int(log.isna().sum().sum()))
print(log.groupby("Patient_File_No").size().unique(), "rows per patient")
log.head()

(30296, 11) | patients: 1082 | missing: 0
[28] rows per patient


,Patient_File_No,Day_of_Cycle,Cycle_Phase,Breast_Soreness_VAS,Painful_Touch_VAS,Pelvic_Pressure_VAS,Bloating_Delta_cm,Fatigue_MFI5_VAS,Acne_Severity_Likert,Hirsutism_mFG_Score,PCOS_Status_Label
0,1,1,Menstrual,3.54,4.72,4.78,2.15,7.28,3,15,1
1,1,2,Menstrual,2.39,3.75,5.57,2.20,5.48,1,16,1
2,1,3,Menstrual,3.42,3.98,9.12,4.06,6.03,2,16,1
3,1,4,Menstrual,2.68,3.62,4.72,2.32,6.91,2,16,1
4,1,5,Menstrual,1.52,6.79,5.85,2.09,6.42,2,15,1


## 2. Build one row per patient
For each of the 7 symptom measures: overall mean / std / max, the mean in each cycle phase, and the luteal-minus-follicular and menstrual-minus-follicular differences (cyclical patterns matter for PMDD and dysmenorrhea).

In [3]:
SYMPTOMS = ["Breast_Soreness_VAS", "Painful_Touch_VAS", "Pelvic_Pressure_VAS", "Bloating_Delta_cm",
            "Fatigue_MFI5_VAS", "Acne_Severity_Likert", "Hirsutism_mFG_Score"]
g = log.groupby("Patient_File_No")

feat = pd.concat([g[SYMPTOMS].mean().add_suffix("_mean"),
                  g[SYMPTOMS].std().add_suffix("_std"),
                  g[SYMPTOMS].max().add_suffix("_max")], axis=1)

phase = log.pivot_table(index="Patient_File_No", columns="Cycle_Phase", values=SYMPTOMS, aggfunc="mean")
phase.columns = [f"{s}_{p}" for s, p in phase.columns]
feat = feat.join(phase)

for s in SYMPTOMS:
    feat[f"{s}_Luteal_minus_Follicular"]   = feat[f"{s}_Luteal"]    - feat[f"{s}_Follicular"]
    feat[f"{s}_Menstrual_minus_Follicular"] = feat[f"{s}_Menstrual"] - feat[f"{s}_Follicular"]

feat = feat.reset_index()
print(feat.shape)
feat.head()

(1082, 64)


,Patient_File_No,Breast_Soreness_VAS_mean,Painful_Touch_VAS_mean,Pelvic_Pressure_VAS_mean,Bloating_Delta_cm_mean,Fatigue_MFI5_VAS_mean,Acne_Severity_Likert_mean,Hirsutism_mFG_Score_mean,Breast_Soreness_VAS_std,Painful_Touch_VAS_std,...,Pelvic_Pressure_VAS_Luteal_minus_Follicular,Pelvic_Pressure_VAS_Menstrual_minus_Follicular,Bloating_Delta_cm_Luteal_minus_Follicular,Bloating_Delta_cm_Menstrual_minus_Follicular,Fatigue_MFI5_VAS_Luteal_minus_Follicular,Fatigue_MFI5_VAS_Menstrual_minus_Follicular,Acne_Severity_Likert_Luteal_minus_Follicular,Acne_Severity_Likert_Menstrual_minus_Follicular,Hirsutism_mFG_Score_Luteal_minus_Follicular,Hirsutism_mFG_Score_Menstrual_minus_Follicular
0,1,3.399643,2.275714,2.932857,1.204286,4.130357,1.821429,14.714286,2.343986,1.655298,...,0.054135,3.85675,-0.222596,1.58525,2.310288,4.31275,0.625000,0.625,-1.557692,0.350
1,2,2.302143,0.622500,1.776786,0.802857,3.009643,0.178571,2.821429,2.353455,0.823797,...,0.763558,3.64525,0.055673,2.29475,0.772308,2.18400,0.182692,-0.125,-0.326923,-1.450
2,3,3.286429,2.382500,3.248929,1.313214,5.928214,1.678571,16.571429,2.422778,1.509470,...,0.528750,4.34275,0.485962,1.48550,2.065481,4.20625,-0.384615,-1.000,-0.163462,0.975
3,4,3.467857,1.905714,2.980714,1.284643,6.053929,1.714286,11.107143,2.406933,1.502510,...,0.529327,4.20425,-0.031923,1.98100,1.226827,2.83975,0.605769,-0.825,-2.394231,-2.625
4,5,2.928214,1.289286,1.761071,0.880357,4.146786,0.321429,4.000000,1.539577,1.184408,...,0.657404,3.97525,0.226250,1.22225,2.504615,2.93600,0.057692,0.350,0.317308,-2.975


## 3. Join targets

In [4]:
lab = pd.read_csv(LABEL)
target_cols = ["Patient_File_No"] + [f"{d}_{s}" for d in DISEASES for s in ("ActualFlag", "ActualScore")]
data = feat.merge(lab[target_cols], on="Patient_File_No", how="inner", validate="1:1")
assert len(data) == len(feat) == len(lab), "patient mismatch between files"

FEATURES = [c for c in feat.columns if c != "Patient_File_No"]   # no PCOS label, no Actual* columns
X = data[FEATURES]
print(len(FEATURES), "features |", len(data), "patients")
pd.DataFrame({"Disease": DISEASES,
              "N_positive": [int(data[f"{d}_ActualFlag"].sum()) for d in DISEASES],
              "Prevalence": [round(data[f"{d}_ActualFlag"].mean(), 4) for d in DISEASES]})

63 features | 1082 patients


,Disease,N_positive,Prevalence
0,Infertility,508,0.4695
1,Dysmenorrhea,730,0.6747
2,PMDD,447,0.4131
3,T2D,192,0.1774
4,CVD,3,0.0028
5,Endometrial,11,0.0102


## 4. Severity bands
- **Classification (probability):** Low < 0.33, Moderate 0.33-0.66, High >= 0.66
- **Regression (risk score):** Minimal < 0.2, Mild 0.2-0.4, Moderate 0.4-0.6, Severe >= 0.6

In [5]:
def clf_category(p):
    return pd.cut(p, [-np.inf, .33, .66, np.inf], labels=["Low", "Moderate", "High"]).astype(str)

def reg_category(s):
    return pd.cut(s, [-np.inf, .2, .4, .6, np.inf], labels=["Minimal", "Mild", "Moderate", "Severe"]).astype(str)

## 5. Train, cross-validate and save
Folds are capped at the number of positives (CVD has only 3). Classifiers use `scale_pos_weight` for class imbalance.

In [6]:
clf_table = pd.DataFrame({"Patient_File_No": data.Patient_File_No})
reg_table = clf_table.copy()
metrics, importances = [], {}
bundle = {"features": FEATURES, "classifiers": {}, "regressors": {},
          "clf_bins": "Low<0.33<=Moderate<0.66<=High",
          "reg_bins": "Minimal<0.2<=Mild<0.4<=Moderate<0.6<=Severe",
          "note": "Input = per-patient aggregates of the daily symptom log (see notebook section 2)."}

for d in DISEASES:
    y_c = data[f"{d}_ActualFlag"]
    y_r = data[f"{d}_ActualScore"].clip(0, 1)
    n_pos = int(y_c.sum())
    k = max(2, min(5, n_pos))
    spw = (len(y_c) - n_pos) / max(n_pos, 1)

    clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                        scale_pos_weight=spw, eval_metric="logloss", random_state=SEED, n_jobs=-1)
    reg = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                       objective="reg:squarederror", random_state=SEED, n_jobs=-1)

    p = cross_val_predict(clf, X, y_c, cv=StratifiedKFold(k, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    s = np.clip(cross_val_predict(reg, X, y_r, cv=KFold(k, shuffle=True, random_state=SEED)), 0, 1)
    pred = (p >= .5).astype(int)

    clf_table[f"{d}_Actual"]   = y_c.values
    clf_table[f"{d}_Prob"]     = p.round(4)
    clf_table[f"{d}_Pred"]     = pred
    clf_table[f"{d}_Severity"] = clf_category(p)
    reg_table[f"{d}_ActualScore"] = y_r.values.round(4)
    reg_table[f"{d}_RiskScore"]   = s.round(4)
    reg_table[f"{d}_Severity"]    = reg_category(s)

    metrics.append({"Disease": d, "N_positive": n_pos, "CV_folds": k,
                    "AUC": roc_auc_score(y_c, p), "PR_AUC": average_precision_score(y_c, p),
                    "F1": f1_score(y_c, pred, zero_division=0), "BalancedAcc": balanced_accuracy_score(y_c, pred),
                    "Reg_R2": r2_score(y_r, s), "Reg_MAE": mean_absolute_error(y_r, s),
                    "Reg_RMSE": mean_squared_error(y_r, s) ** .5})

    clf.fit(X, y_c); reg.fit(X, y_r)       # final models on all patients
    importances[d] = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=False)
    bundle["classifiers"][d] = clf
    bundle["regressors"][d] = reg
    for kind, m in (("classifier", clf), ("regressor", reg)):
        with open(f"{OUT}/models/{d}_{kind}.pkl", "wb") as f:
            pickle.dump(m, f)
        joblib.dump(m, f"{OUT}/models/{d}_{kind}.joblib")
    print("done:", d)

with open(f"{OUT}/models/all_models_bundle.pkl", "wb") as f:
    pickle.dump(bundle, f)
joblib.dump(bundle, f"{OUT}/models/all_models_bundle.joblib")

done: Infertility


done: Dysmenorrhea


done: PMDD


done: T2D


done: CVD


done: Endometrial


['/mnt/user-data/outputs/symptom_log_model/models/all_models_bundle.joblib']

## 6. Cross-validated metrics

In [7]:
met = pd.DataFrame(metrics).round(4)
met

,Disease,N_positive,CV_folds,AUC,PR_AUC,F1,BalancedAcc,Reg_R2,Reg_MAE,Reg_RMSE
0,Infertility,508,5,1.0000,1.0000,0.9990,0.9990,0.9920,0.0189,0.0233
1,Dysmenorrhea,730,5,0.4733,0.6572,0.6849,0.4892,-0.0639,0.0761,0.0906
2,PMDD,447,5,0.4858,0.4078,0.4036,0.4967,-0.1003,0.0944,0.1123
3,T2D,192,5,0.9963,0.9858,0.9207,0.9581,0.9908,0.0052,0.0106
4,CVD,3,3,0.8706,0.1247,0.0000,0.5000,0.9824,0.0035,0.0091
5,Endometrial,11,5,0.9868,0.5891,0.4706,0.6809,0.9886,0.0045,0.0060


## 7. Infertility in detail

In [8]:
cm = confusion_matrix(clf_table.Infertility_Actual, clf_table.Infertility_Pred)
print("Infertility confusion matrix (rows = actual, cols = predicted)")
print(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))
print("\nTop 10 features (classifier):")
importances["Infertility"].head(10).round(4)

Infertility confusion matrix (rows = actual, cols = predicted)
          Pred 0  Pred 1
Actual 0     574       0
Actual 1       1     507

Top 10 features (classifier):


Hirsutism_mFG_Score_mean          0.4090
Hirsutism_mFG_Score_Follicular    0.4049
Hirsutism_mFG_Score_Luteal        0.1861
Pelvic_Pressure_VAS_mean          0.0000
Bloating_Delta_cm_mean            0.0000
Painful_Touch_VAS_mean            0.0000
Breast_Soreness_VAS_mean          0.0000
Breast_Soreness_VAS_std           0.0000
Painful_Touch_VAS_std             0.0000
Pelvic_Pressure_VAS_std           0.0000
dtype: float32

## 8. Per-patient table: Classification (XGBClassifier)

In [9]:
clf_cols = ["Patient_File_No"] + [f"{d}_{s}" for d in DISEASES for s in ("Prob", "Severity")]
clf_table[clf_cols].head(20)

,Patient_File_No,Infertility_Prob,Infertility_Severity,Dysmenorrhea_Prob,Dysmenorrhea_Severity,PMDD_Prob,PMDD_Severity,T2D_Prob,T2D_Severity,CVD_Prob,CVD_Severity,Endometrial_Prob,Endometrial_Severity
0,1,0.9974,High,0.2411,Low,0.3813,Moderate,0.0015,Low,0.0019,Low,0.0002,Low
1,2,0.0025,Low,0.4602,Moderate,0.1532,Low,0.0001,Low,0.0010,Low,0.0002,Low
2,3,0.9975,High,0.7321,High,0.1847,Low,0.9995,High,0.0153,Low,0.0003,Low
3,4,0.9975,High,0.3846,Moderate,0.2886,Low,0.9988,High,0.0019,Low,0.0003,Low
4,5,0.0025,Low,0.6960,High,0.7470,High,0.0001,Low,0.0005,Low,0.0001,Low
5,6,0.9975,High,0.8284,High,0.5019,Moderate,0.0035,Low,0.0019,Low,0.0007,Low
6,7,0.9974,High,0.7998,High,0.3782,Moderate,0.8479,High,0.0006,Low,0.0007,Low
7,8,0.9975,High,0.2696,Low,0.5033,Moderate,0.0006,Low,0.0006,Low,0.0003,Low
8,9,0.0025,Low,0.4194,Moderate,0.3574,Moderate,0.0001,Low,0.0008,Low,0.0002,Low
9,10,0.9974,High,0.7495,High,0.5254,Moderate,0.9871,High,0.0033,Low,0.0008,Low


## 9. Per-patient table: Regression (XGBRegressor, risk score 0-1)

In [10]:
reg_cols = ["Patient_File_No"] + [f"{d}_{s}" for d in DISEASES for s in ("RiskScore", "Severity")]
reg_table[reg_cols].head(20)

,Patient_File_No,Infertility_RiskScore,Infertility_Severity,Dysmenorrhea_RiskScore,Dysmenorrhea_Severity,PMDD_RiskScore,PMDD_Severity,T2D_RiskScore,T2D_Severity,CVD_RiskScore,CVD_Severity,Endometrial_RiskScore,Endometrial_Severity
0,1,0.6684,Severe,0.3590,Mild,0.3696,Mild,0.3547,Mild,0.2582,Mild,0.2849,Mild
1,2,0.0889,Minimal,0.4218,Moderate,0.3697,Mild,0.1782,Minimal,0.1805,Minimal,0.1987,Minimal
2,3,0.6851,Severe,0.4179,Moderate,0.4624,Moderate,0.4427,Moderate,0.3349,Mild,0.2776,Mild
3,4,0.6065,Severe,0.3804,Mild,0.3333,Mild,0.4281,Moderate,0.3380,Mild,0.2734,Mild
4,5,0.1147,Minimal,0.4057,Moderate,0.3907,Mild,0.2396,Mild,0.2236,Mild,0.1754,Minimal
5,6,0.6518,Severe,0.4172,Moderate,0.3620,Mild,0.3606,Mild,0.2850,Mild,0.2829,Mild
6,7,0.6704,Severe,0.4489,Moderate,0.4118,Moderate,0.4122,Moderate,0.3034,Mild,0.2939,Mild
7,8,0.6472,Severe,0.3700,Mild,0.3746,Mild,0.3481,Mild,0.2717,Mild,0.3099,Mild
8,9,0.0987,Minimal,0.3862,Mild,0.3094,Mild,0.1495,Minimal,0.1500,Minimal,0.1654,Minimal
9,10,0.6245,Severe,0.4339,Moderate,0.4409,Moderate,0.3995,Mild,0.3137,Mild,0.3128,Mild


## 10. Severity distribution

In [11]:
dist = []
for d in DISEASES:
    for name, tbl in (("Classification", clf_table), ("Regression", reg_table)):
        dist.append({"Disease": d, "Model": name, **tbl[f"{d}_Severity"].value_counts().to_dict()})
pd.DataFrame(dist).fillna(0).set_index(["Disease", "Model"]).astype(int)

Low  High  Minimal  Severe  Moderate  Mild
Disease      Model                                                      
Infertility  Classification   575   507        0       0         0     0
             Regression         0     0      557     385       123    17
Dysmenorrhea Classification    68   388        0       0       626     0
             Regression         0     0        0       0       600   482
PMDD         Classification   252   122        0       0       708     0
             Regression         0     0        0       0       578   504
T2D          Classification   870   194        0       0        18     0
             Regression         0     0      357       0       193   532
CVD          Classification  1082     0        0       0         0     0
             Regression         0     0      389       0         0   693
Endometrial  Classification  1074     5        0       0         3     0
             Regression         0     0      318       0         0   764

## 11. Export tables

In [12]:
clf_table.to_csv(f"{OUT}/classification_predictions_per_patient.csv", index=False)
reg_table.to_csv(f"{OUT}/regression_predictions_per_patient.csv", index=False)
met.to_csv(f"{OUT}/model_metrics.csv", index=False)
data[["Patient_File_No"] + FEATURES].to_csv(f"{OUT}/patient_level_features.csv", index=False)

with pd.ExcelWriter(f"{OUT}/per_patient_predictions.xlsx", engine="openpyxl") as w:
    clf_table.to_excel(w, sheet_name="Classification", index=False)
    reg_table.to_excel(w, sheet_name="Regression", index=False)
    met.to_excel(w, sheet_name="Metrics", index=False)
    for ws in w.book.worksheets:
        ws.freeze_panes = "B2"
        for c in ws[1]:
            c.font = Font(bold=True, name="Arial")
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = max(12, len(str(col[0].value)) + 2)
print("exported to", OUT)

exported to /mnt/user-data/outputs/symptom_log_model


## 12. Reload check and how to score new patients

In [13]:
b_job = joblib.load(f"{OUT}/models/all_models_bundle.joblib")
with open(f"{OUT}/models/all_models_bundle.pkl", "rb") as f:
    b_pkl = pickle.load(f)

sample = data[b_job["features"]].head(3)
for d in DISEASES:
    assert np.allclose(b_job["classifiers"][d].predict_proba(sample), b_pkl["classifiers"][d].predict_proba(sample)), d
    assert np.allclose(b_job["regressors"][d].predict(sample), b_pkl["regressors"][d].predict(sample)), d
print("pickle and joblib bundles give identical predictions")

# New patients: run the section-2 aggregation on their 28-day log, then:
#   prob  = b_job["classifiers"]["Infertility"].predict_proba(new_feat[b_job["features"]])[:, 1]
#   score = b_job["regressors"]["Infertility"].predict(new_feat[b_job["features"]]).clip(0, 1)

pickle and joblib bundles give identical predictions
